<a href="https://colab.research.google.com/github/Induwara427/Statistical-Learning-e21427/blob/main/E21427_Assignment_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Question 1: The Total Variance Illusion (PCA Subplots 1 & 2 vs. FA Subplots 1 & 2) — Comprehensive Analysis



#### 1. Physical Nature of the Data from $\text{Sensor}_4$ (Uniqueness $\varphi^2 > 98\%$)

In a Factor Analysis (FA) framework, the total variance of any standardized sensor channel $j$ is decomposed into two orthogonal components:

$$
\text{Var}(Z_j) = h_j^2 + \varphi_j^2 = 1
$$

where $h_j^2$ represents the communality (shared variance explained by latent global structural or operational factors $\mathbf{F}$), and $\varphi_j^2$ represents the uniqueness (sensor-specific variance plus measurement noise).

A Uniqueness value exceeding 98% indicates:

- Decoupling from system dynamics: Nearly all variability in $\text{Sensor}_4$ is independent of the underlying physical or mechanical processes affecting the rest of the system.
- Dominance of idiosyncratic effects: The signal is primarily composed of localized noise, sensor malfunction, calibration drift, or external interference unique to that channel.



#### 2. How Massive Localized Noise "Tricks" the PCA Engine

Principal Component Analysis (PCA) is a geometric method that identifies orthogonal directions maximizing total variance, defined by:

$$
\text{tr}(\mathbf{S}) = \sum_{j=1}^{m} \lambda_j
$$

PCA is blind to the *origin* of variance and treats all variability as equally informative, regardless of whether it is:

1. Shared structural variance: arising from correlated system-wide behavior.
2. Idiosyncratic variance: arising from isolated sensor noise.

Because $\text{Sensor}_4$ exhibits disproportionately large variance (e.g., $\sigma^2 \approx 1.96$), it dominates the covariance structure. As a result, PCA aligns its leading eigenvectors with this high-variance direction.

This produces:

- Artificial inflation of leading eigenvalues ($\lambda_1, \lambda_2$)
- Apparent “strong structure” in the principal components
- A misleading interpretation that the system has fewer, more dominant latent modes than it actually does

In reality, PCA is primarily capturing **noise energy concentration**, not meaningful system-wide dynamics.



#### 3. Operational Risks of Relying Solely on PCA for Anomaly Detection

Using PCA alone for monitoring introduces several failure modes:

- False alarms (Type I errors):  
  Noise spikes in $\text{Sensor}_4$ strongly perturb principal component scores, potentially triggering Hotelling’s $T^2$ alarms even when the underlying process is stable.

- Missed detections (Type II errors):  
  The inflated variance structure broadens the normal operating subspace, masking subtle but physically meaningful shifts in latent process factors (e.g., early-stage degradation modes).

- Incorrect dimensionality selection:  
  Choosing $k$ based on cumulative explained variance (e.g., $k=3$ for 90% variance) risks embedding noise-dominated directions into the retained subspace, contaminating both monitoring statistics (e.g., $T^2$ and SPE/Q) and long-term baseline models.





### Question 2: Decoupling Structural Loading via Rotation (FA Subplot 1 vs. PCA Eigenvectors)



#### 1. How Varimax Rotation Relaxes PCA Rules to Create a "Simple Structure"

Principal Component Analysis (PCA) enforces a strict optimization hierarchy: each principal component is extracted sequentially to maximize remaining variance under orthogonality constraints. This produces eigenvalues ordered as $\lambda_1 > \lambda_2 > \dots > \lambda_m$, and the first component typically absorbs variance across many variables simultaneously.

As a consequence, PCA produces diffuse loading structures, where most variables contribute non-trivially to each principal component. This leads to cross-loadings, meaning each component becomes a mixture of multiple underlying physical mechanisms, reducing interpretability.

Factor Analysis (FA) with Varimax rotation relaxes this constraint by performing an orthogonal rotation within the retained $k$-dimensional latent subspace. Instead of changing the explained variance, it redistributes that variance across factors to improve interpretability.

The Varimax criterion maximizes:

$$
\sum_{r=1}^{k} \left[ \frac{1}{m} \sum_{j=1}^{m} \lambda_{j,r}^4 - \left( \frac{1}{m} \sum_{j=1}^{m} \lambda_{j,r}^2 \right)^2 \right]
$$

This objective:

- Encourages sparse loadings, pushing values toward $0$ or $\pm 1$
- Reduces intermediate cross-loadings
- Produces Kaiser’s simple structure, where each factor is dominated by a small subset of variables

Unlike PCA, Varimax does not prioritize sequential variance maximization; instead, it optimizes interpretability within the same subspace.



#### 2. The Plant Operator’s Perspective: Troubleshooting Advantages of Rotated FA vs. Raw PCA

From an industrial monitoring perspective, unrotated PCA outputs are often difficult to interpret because each principal component mixes multiple physical phenomena:

- A single component may combine structural loading, thermal drift, and sensor noise
- An alarm on $\text{PC}_1$ provides no direct mapping to a specific subsystem
- Diagnostic effort requires investigating all sensors simultaneously

This creates ambiguity in fault localization.

Varimax-rotated Factor Analysis resolves this by aligning latent factors with coherent physical groupings:

- Factor 1: Strong loadings from Sensor\_1 and Sensor\_2 (primary structural mode)
- Factor 2: Dominated by Sensor\_3 (secondary operational mode)

This structure enables direct interpretability:

- A spike in Factor 1 indicates a disturbance localized to the structural subsystem associated with Sensors 1 and 2
- A spike in Factor 2 isolates issues to the subsystem represented by Sensor 3





### Question 3: Determining Subspace Truncation ($k$) using $T^2$ and $Q$ Profiles



#### 1. Behavior of the Mean $Q$ Statistic (SPE) Curve Across Cutoffs

The $Q$ statistic, or Squared Prediction Error (SPE), measures the squared orthogonal distance between an observation $\mathbf{x}$ and its projection onto the retained $k$-dimensional principal subspace. It quantifies the portion of variance not explained by the model.

$$
Q = \|\mathbf{x} - \hat{\mathbf{x}}\|^2 = \sum_{j=k+1}^{m} e_j^2
$$

As the truncation parameter $k$ increases, the Mean $Q$ curve exhibits two distinct regimes:

- From $k=1$ to $k=2$: The curve shows a sharp decline.  
  The first component captures only the dominant physical mode (e.g., primary structural loading affecting $\text{Sensor}_1$ and $\text{Sensor}_2$), leaving the secondary operational mode (e.g., $\text{Sensor}_3$ dynamics) largely unexplained. Including a second component significantly reduces reconstruction error.

- From $k=2$ to $k=3$: The curve becomes flat (elbow region).  
  The additional component contributes negligibly to reducing reconstruction error, indicating that no additional structured correlation remains in the system.



#### 2. Why the Flat Elbow Identifies $k=2$ as the True Dimensionality

The true latent dimensionality of an industrial system corresponds to the number of independent physical processes driving correlated behavior across sensors.

- Low-rank structure capture:  
  The first two components capture the dominant correlated dynamics in the system. Their inclusion sharply reduces $Q$, reflecting removal of structured error.

- Noise floor regime:  
  After $k=2$, remaining variance is dominated by uncorrelated sensor-specific noise. This residual does not exhibit coherent structure across variables.

- Elbow interpretation:  
  The transition from large error reduction (structured signal) to negligible reduction (noise) produces a distinct elbow. This indicates that $k=2$ captures the full effective physical rank of the system, while higher components primarily model noise.



#### 3. Impact of an Incorrect Truncation ($k=3$)

Selecting $k=3$ instead of the true dimensionality introduces structural and diagnostic degradation:

- Contamination of the principal subspace:  
  A noise-dominated direction (often associated with high-variance sensors such as $\text{Sensor}_4$) is incorporated into the model, embedding non-physical variability into the “healthy” subspace.

- Distortion of baseline representation:  
  The retained PCA model no longer reflects purely physical dynamics; it becomes partially driven by idiosyncratic sensor noise.

- Loss of sensitivity in the residual space ($Q$):  
  Because noise is absorbed into the principal subspace, the residual space becomes artificially clean. The $Q$ statistic loses sensitivity to sensor faults, calibration drift, or sudden failures.

- Increased false alarms in $T^2$ space:  
  Faults that should appear in the residual domain may instead propagate into the principal subspace, destabilizing Hotelling’s $T^2$ monitoring statistic and triggering misleading system-wide alerts.



## Question 4: Operational Trade-offs in System Health Monitoring — Comprehensive Analysis

### 1. Operational Comparison of the Two Monitoring Strategies

| Operational Metric | The PCA Strategy (Hotelling's $T^2$ + $Q$ Statistic) | The FA Strategy (Rotated Latent Factor Scores) |
| :--- | :--- | :--- |
| Mathematical Objective | Maximizes total global variance projection across orthogonal axes. | Explains shared covariance via a low-rank latent structure while isolating channel-specific variance. |
| Sensitivity to Global Structural Anomalies | High. Captures any shift that introduces or modifies large-scale energy signatures in the system. | High. Directly tracks changes in the true underlying physical and mechanical mechanisms. |
| Diagnostic Resolution | Low (Fault Smearing). Because eigenvectors are linear combinations of all sensors, a fault in one channel bleeds across multiple principal components. | High (Phenomenological Isolation). Varimax rotation aligns factors with specific physical subsystems, confining anomalies to individual indicators. |
| False Alarm Resiliency | Low. Vulnerable to localized sensor instabilities, which can easily trigger a global $T^2$ alarm. | High. Disregards individual sensor anomalies, preserving the integrity of system-level health tracking. |



### 2. Selection and Justification: The Superior Robustness of the FA Strategy

The Factor Analysis (FA) Strategy is significantly more robust against a single sensor losing calibration or experiencing an electrical short.

This resilience is mathematically driven by how the **Sensor Uniqueness Noise Floor ($\varphi^2$)** is integrated into the model, contrasting sharply with the structural limitations of PCA.

####  Mathematical Justification via the Uniqueness Tensor ($\mathbf{\Psi}$)

In Factor Analysis, the standardized observation vector $\mathbf{z}$ is modeled as:

$$\mathbf{z} = \mathbf{\Lambda}\mathbf{F} + \boldsymbol{\epsilon}$$

Where:
* $\mathbf{\Lambda}$ is the factor loading matrix.
* $\mathbf{F}$ represents the latent factors.
* $\boldsymbol{\epsilon}$ is the vector of unique/specific errors.

The covariance matrix of $\boldsymbol{\epsilon}$ is explicitly defined as the diagonal uniqueness matrix:

$$\mathbf{\Psi} = \text{diag}(\varphi_1^2, \varphi_2^2, \dots, \varphi_m^2)$$

When computing real-time latent factor scores ($\hat{\mathbf{F}}$) from incoming sensor data, standard extraction techniques (such as Thomson’s Regression Method) utilize the following formulation:

$$\hat{\mathbf{F}} = (\mathbf{\Lambda}^T \mathbf{\Psi}^{-1} \mathbf{\Lambda} + \mathbf{I})^{-1} \mathbf{\Lambda}^T \mathbf{\Psi}^{-1} \mathbf{z}$$

The presence of the inverse uniqueness matrix, $\mathbf{\Psi}^{-1} = \text{diag}\left(\frac{1}{\varphi_1^2}, \frac{1}{\varphi_2^2}, \dots, \frac{1}{\varphi_m^2}\right)$, provides an automated mathematical safeguard:

* **Automated Noise De-weighting:** If a channel like $\text{Sensor}_4$ has an incredibly high uniqueness value ($\varphi_4^2 > 98\%$), its corresponding weight in the factor scoring matrix scales by $\frac{1}{\varphi_4^2} \approx \frac{1}{0.98} \approx 1.02$. In contrast, a highly reliable structural sensor with a low uniqueness ($\varphi_1^2 = 0.05$) receives a weight scaled by $\frac{1}{0.05} = 20$. Consequently, if $\text{Sensor}_4$ experiences an electrical short or completely fails, its capacity to corrupt the latent factor scores ($\mathbf{F}$) is mathematically suppressed to near zero.
* **Containment of Single-Channel Variance Spikes:** When any sensor experiences a loss of calibration or a short circuit, it injects a massive burst of idiosyncratic, uncorrelated variance into that single channel. Because the FA architecture restricts the factor scores to capture only shared covariance, this sudden surge in single-channel variance cannot pass through the loading matrix $\mathbf{\Lambda}$. Instead, the model automatically shunts the entire fault signature directly into that sensor's unique error term ($\epsilon_j$). The latent factor scores tracking the overall health of the asset remain completely undisturbed.



####  Why the PCA Strategy Fails in This Scenario (Fault Smearing)

Principal Component Analysis lacks a diagonal uniqueness tensor ($\mathbf{\Psi}$). It operates under the rigid assumption that all variance is isotropic and structurally relevant.

When a single sensor shorts out, it creates a massive spike in total system variance. Because the principal components ($\mathbf{P}$) are forced to maximize global variance, this localized fault:
1. Generates a massive shift in the principal coordinate system, pulling the leading principal axes toward the failing sensor's coordinate direction.
2. Causes the fault energy to leak into the principal scores, destabilizing Hotelling's $T^2$ metric.

As a result, a plant operator relying on PCA will see a critical breach in the $T^2$ control limit, indicating a major "global structural anomaly." In reality, the asset's structural integrity is perfectly sound—the system has simply been misled by a single faulty instrument. The FA framework avoids this diagnostic error by cleanly separating sensor-specific faults from true, asset-wide mechanical degradation.